### BART

In [3]:
from neurolex.config import MODELS, COLORS
from transformers import pipeline  

cfg = MODELS["classifier"]
cfg

c:\code\projects\NEUROLEX\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'model_name': 'facebook/bart-large-mnli',
 'task': 'zero-shot-classification',
 'description': 'Multi-label zero-shot classification via BART-MNLI'}

In [13]:
classifier = pipeline(task=cfg["task"], model=cfg["model_name"], device=0)
classifier

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 829.77it/s, Materializing param=model.shared.weight]                                   


ZeroShotClassificationPipeline: {'model': 'BartForSequenceClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}

In [16]:
text = "today i learn about Bideractional Encoder Representations from Transformers (BERT) and how it can be used for text classification tasks."
labels = ["education", "sports", "politics", "technology"]
multi_label = False
classifier(text, candidate_labels=labels, multi_label=multi_label)

{'sequence': 'today i learn about Bideractional Encoder Representations from Transformers (BERT) and how it can be used for text classification tasks.',
 'labels': ['technology', 'education', 'sports', 'politics'],
 'scores': [0.8080320358276367,
  0.1824803650379181,
  0.0054015712812542915,
  0.004086027853190899]}

In [20]:
text = "today i learn about Bideractional Encoder Representations from Transformers (BERT) and how it can be used for text classification tasks."
labels = ["NLP","CNN", "RL", "ML"]
multi_label = True
output =classifier(text, candidate_labels=labels, multi_label=multi_label,hypothesis_template="This news is about {}.",)

In [23]:
dict(zip(output["labels"], output["scores"]))


{'ML': 0.0278380885720253,
 'NLP': 0.008218478411436081,
 'RL': 0.0010877501918002963,
 'CNN': 0.0002896134683396667}

### NER

In [4]:
cfg = MODELS["ner"]
ner = pipeline(
        cfg["task"],
        model=cfg["model_name"],
        aggregation_strategy=cfg["aggregation"],
    )
ner

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 701.79it/s, Materializing param=classifier.weight]                                      
BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TokenClassificationPipeline: {'model': 'BertForTokenClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}

In [ ]:
result = ner("OpenAI, founded in San Francisco, released GPT-4 as a breakthrough in artificial intelligence.")
result

In [ ]:
[
            {
                "entity": r["entity_group"],
                "word": r["word"],
                "score": round(r["score"], 4),
                "start": r["start"],
                "end": r["end"],
                # "color": seENTITY_COLORS.get(r["entity_group"], "#8B949E"),
            }
            for r in result
        ]

In [13]:
entity_name = "BERT"
url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{entity_name.replace(' ', '_')}"
url

'https://en.wikipedia.org/api/rest_v1/page/summary/BERT'

In [18]:
def link_entity( entity_name: str) -> dict:
        """
        Link entity to Wikipedia using summary API.

        Returns:
            dict with title, summary, url, thumbnail
        """
        import requests

        url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{entity_name.replace(' ', '_')}"
        try:
            resp = requests.get(url, timeout=8)
            print(resp)
            if resp.status_code == 200:
                data = resp.json()
                return {
                    "title": data.get("title", entity_name),
                    "summary": data.get("extract", "No description found.")[:400],
                    "url": data.get("content_urls", {}).get("desktop", {}).get("page", ""),
                    "thumbnail": data.get("thumbnail", {}).get("source", ""),
                    "found": True,
                }
        except Exception:
            pass
        return {"title": entity_name, "summary": "Entity not found in Wikipedia.", "found": False}

In [19]:
link_entity("OPENAI")

<Response [403]>


{'title': 'OPENAI',
 'summary': 'Entity not found in Wikipedia.',
 'found': False}

In [ ]:
# how to use counter
co